In [9]:
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("BASE_PATH"))

D:/Aus-weather-analysis


In [10]:
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("BASE_PATH"))

D:/Aus-weather-analysis


In [11]:
from pathlib import Path

print(Path.cwd())
print((Path.cwd().parent / ".env").exists())

d:\Aus-weather-analysis\notebooks
True


In [12]:
from pathlib import Path

BASE_PATH = Path.cwd().parent

print(BASE_PATH)
print(list((BASE_PATH / "data" / "raw").iterdir()))

d:\Aus-weather-analysis
[WindowsPath('d:/Aus-weather-analysis/data/raw/Weather Test Data.csv'), WindowsPath('d:/Aus-weather-analysis/data/raw/Weather Training Data.csv')]


In [13]:
from pathlib import Path

BASE_PATH = Path.cwd().parent

print(BASE_PATH)
print((BASE_PATH / "data").exists())
print((BASE_PATH / "data" / "processed").exists())

d:\Aus-weather-analysis
True
True


In [14]:
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os

In [15]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv

# Load Data 
load_dotenv()
BASE_PATH = os.getenv("BASE_PATH")
OUTPUT_PATH = f"{BASE_PATH}/data/processed/clean_weather_training_data.parquet"
df = pd.read_csv(f"{BASE_PATH}/data/raw/Weather Training Data.csv")

# Drop Identifier Column 
df.drop(columns=['row ID'], inplace=True)


# Drop Duplicates
df = df.drop_duplicates()

# Range of Cloud According to Oktas from 0 to 8
df.loc[df['Cloud3pm'] == 9.0, 'Cloud3pm'] = np.nan

# Filling Missing Values With Median
numerical_columns = ['MaxTemp', 'MinTemp', 'Rainfall', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity3pm', 'Humidity9am', 'Pressure9am', 'Pressure3pm', 'Temp9am', 'Temp3pm']
for col in numerical_columns: 
    df[col] = df.groupby('Location')[col].transform(
    lambda x: x.fillna(x.median())
)
    df[col] = df[col].fillna(df[col].median())   # global fallback


# Filling Wind Directions With Mode 
df['WindGustDir'] = df.groupby('Location')['WindGustDir'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x)
)

global_mode = df['WindGustDir'].mode()[0]
df['WindGustDir'] = df['WindGustDir'].fillna(global_mode)

# Filling Columns with Mode
categorical_columns = ['WindDir9am', 'WindDir3pm', 'RainToday']
for col in categorical_columns: 
    df[col] = df.groupby('Location')[col].transform(
        lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x)
)

cloud_columns = ['Cloud9am', 'Cloud3pm']
for col in cloud_columns:
    df[col] = df.groupby('Location')[col].transform(
        lambda x: x.fillna(x.median())
    )
    df[col] = df[col].fillna(df[col].median()) # fallback for locations where all values were NaN (median() returns NaN too)

# Must run after Rainfall is fully imputed above, since RainToday is derived from it
df['RainToday'] = np.where(df['Rainfall'] > 1, 'Yes', 'No')


# Downcasting Datatypes to Float32 for Continuous Value Columns
continuous_columns = df.select_dtypes(include='float').columns
df[continuous_columns] = df[continuous_columns].astype('float32')


# Downcast Clouds to int8
clouds_columns = ['Cloud9am', 'Cloud3pm']

for cloud in clouds_columns:
    df[cloud] = df[cloud].round()  # round first: astype('Int8') truncates instead of rounding
    df[cloud] = df[cloud].astype('Int8')

# Downcast Str to Category
str_columns = df.select_dtypes(include='str').columns
for col in str_columns: 
    df[col] = df[col].astype('category')

# Downcast Rain Tomorrow to Boolean Value
df['RainTomorrow'] = df['RainTomorrow'].astype('bool')

# Save New Dataset
df.to_parquet(OUTPUT_PATH, engine='pyarrow')
print("Done: data_cleaning.py")
print("New Dataset Saved in: data/processed/clean_weather_training_data.parquet")
df.isna().sum()
print(df.columns.tolist())


Done: data_cleaning.py
New Dataset Saved in: data/processed/clean_weather_training_data.parquet
['Location', 'MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustDir', 'WindGustSpeed', 'WindDir9am', 'WindDir3pm', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm', 'RainToday', 'RainTomorrow']


In [16]:
df.duplicated().sum()

np.int64(2)

In [17]:
df = df.drop_duplicates()

In [18]:
for col in df.columns: 
    print(f"Missing Values for {col}: {df[col].isna().sum()}")

Missing Values for Location: 0
Missing Values for MinTemp: 0
Missing Values for MaxTemp: 0
Missing Values for Rainfall: 0
Missing Values for Evaporation: 42499
Missing Values for Sunshine: 47285
Missing Values for WindGustDir: 0
Missing Values for WindGustSpeed: 0
Missing Values for WindDir9am: 0
Missing Values for WindDir3pm: 0
Missing Values for WindSpeed9am: 0
Missing Values for WindSpeed3pm: 0
Missing Values for Humidity9am: 0
Missing Values for Humidity3pm: 0
Missing Values for Pressure9am: 0
Missing Values for Pressure3pm: 0
Missing Values for Cloud9am: 0
Missing Values for Cloud3pm: 0
Missing Values for Temp9am: 0
Missing Values for Temp3pm: 0
Missing Values for RainToday: 0
Missing Values for RainTomorrow: 0


In [19]:
numerical_columns = df.select_dtypes(include='number').columns
for col in numerical_columns: 
    print(f"Min Value for {col}: {df[col].min()}")
    print(f"Max Value for {col}: {df[col].max()}")

Min Value for MinTemp: -8.5
Max Value for MinTemp: 33.900001525878906
Min Value for MaxTemp: -4.099999904632568
Max Value for MaxTemp: 48.099998474121094
Min Value for Rainfall: 0.0
Max Value for Rainfall: 371.0
Min Value for Evaporation: 0.0
Max Value for Evaporation: 86.19999694824219
Min Value for Sunshine: 0.0
Max Value for Sunshine: 14.5
Min Value for WindGustSpeed: 6.0
Max Value for WindGustSpeed: 135.0
Min Value for WindSpeed9am: 0.0
Max Value for WindSpeed9am: 130.0
Min Value for WindSpeed3pm: 0.0
Max Value for WindSpeed3pm: 87.0
Min Value for Humidity9am: 0.0
Max Value for Humidity9am: 100.0
Min Value for Humidity3pm: 0.0
Max Value for Humidity3pm: 100.0
Min Value for Pressure9am: 980.5
Max Value for Pressure9am: 1041.0
Min Value for Pressure3pm: 978.2000122070312
Max Value for Pressure3pm: 1039.5999755859375
Min Value for Cloud9am: 0
Max Value for Cloud9am: 9
Min Value for Cloud3pm: 0
Max Value for Cloud3pm: 8
Min Value for Temp9am: -7.0
Max Value for Temp9am: 40.200000762939

In [20]:
continuous_columns = df.select_dtypes(include='float').columns

df[continuous_columns] = df[continuous_columns].astype('float32')
df.info()

<class 'pandas.DataFrame'>
Index: 99484 entries, 0 to 99515
Data columns (total 22 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   Location       99484 non-null  category
 1   MinTemp        99484 non-null  float32 
 2   MaxTemp        99484 non-null  float32 
 3   Rainfall       99484 non-null  float32 
 4   Evaporation    56985 non-null  float32 
 5   Sunshine       52199 non-null  float32 
 6   WindGustDir    99484 non-null  category
 7   WindGustSpeed  99484 non-null  float32 
 8   WindDir9am     99484 non-null  category
 9   WindDir3pm     99484 non-null  category
 10  WindSpeed9am   99484 non-null  float32 
 11  WindSpeed3pm   99484 non-null  float32 
 12  Humidity9am    99484 non-null  float32 
 13  Humidity3pm    99484 non-null  float32 
 14  Pressure9am    99484 non-null  float32 
 15  Pressure3pm    99484 non-null  float32 
 16  Cloud9am       99484 non-null  Int8    
 17  Cloud3pm       99484 non-null  Int8    
 18  Te

In [21]:
df[df['MinTemp'] > df['MaxTemp']]

,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
12195,Newcastle,13.8,13.0,0.0,NaN,NaN,W,39.0,NW,SE,...,44.0,58.0,1017.700012,1015.299988,8,5,8.8,22.000000,No,False
12565,NorahHead,15.7,14.8,0.0,NaN,NaN,WNW,28.0,SSW,NE,...,76.0,69.0,1018.299988,1016.000000,6,5,18.9,20.799999,No,False
35719,MountGinini,3.2,-1.3,0.0,NaN,NaN,E,37.0,W,E,...,89.0,74.0,1017.700012,1015.299988,6,5,6.8,-1.600000,No,False
35921,MountGinini,3.2,-2.7,0.0,NaN,NaN,WSW,48.0,W,W,...,97.0,97.0,1017.700012,1015.299988,6,5,-4.2,-2.900000,No,True
35937,MountGinini,3.2,2.4,0.0,NaN,NaN,WNW,33.0,W,W,...,89.0,69.0,1017.700012,1015.299988,6,5,6.8,10.300000,No,False
35938,MountGinini,3.2,2.9,0.0,NaN,NaN,S,26.0,W,SSW,...,89.0,30.0,1017.700012,1015.299988,6,5,6.8,2.500000,No,False
36150,MountGinini,3.2,0.1,0.0,NaN,NaN,ESE,46.0,W,SSE,...,89.0,93.0,1017.700012,1015.299988,6,5,6.8,-0.100000,No,False
36373,MountGinini,3.2,0.0,0.0,NaN,NaN,S,56.0,W,SSW,...,89.0,69.0,1017.700012,1015.299988,6,5,6.8,10.300000,No,True
36406,MountGinini,3.2,1.0,0.0,NaN,NaN,SW,44.0,W,WSW,...,89.0,100.0,1017.700012,1015.299988,6,5,6.8,0.100000,No,True


In [22]:
categorical_columns = df.select_dtypes(include="str").columns
for col in categorical_columns: 
    print(f"Unique Values for {col}: {df[col].nunique()}")

In [23]:
for col in categorical_columns:  # type: ignore
    print(df[col].unique().tolist())
    print(20 * '-')

In [24]:
df['Location'].value_counts().sort_index(ascending=True)

Location
Adelaide            2178
Albany              2051
Albury              2142
AliceSprings        2119
BadgerysCreek       2041
Ballarat            2122
Bendigo             2110
Brisbane            2202
Cairns              2101
Canberra            2393
Cobar               2090
CoffsHarbour        2066
Dartmoor            2067
Darwin              2217
GoldCoast           2057
Hobart              2239
Katherine           1065
Launceston          2072
Melbourne           1695
MelbourneAirport    2139
Mildura             2124
Moree               2020
MountGambier        2140
MountGinini         2025
Newcastle           2064
Nhil                1136
NorahHead           2028
NorfolkIsland       2038
Nuriootpa           2110
PearceRAAF          1953
Penrith             2059
Perth               2262
PerthAirport        2167
Portland            2113
Richmond            2060
Sale                2093
SalmonGums          2031
Sydney              2361
SydneyAirport       2100
Townsville      

In [25]:
numeric_cols = df.select_dtypes(include='number')
outlier_summary = {}

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary[col] = count

pd.Series(outlier_summary).sort_values(ascending=False)

Rainfall         20043
WindGustSpeed     3778
Pressure9am       1907
WindSpeed3pm      1717
Pressure3pm       1520
Evaporation       1367
WindSpeed9am      1203
Humidity9am        978
Temp3pm            534
MaxTemp            326
Temp9am            209
MinTemp             47
Sunshine             0
Humidity3pm          0
Cloud3pm             0
Cloud9am             0
dtype: int64

In [26]:
print(df['MinTemp'].min())
print(df['MaxTemp'].max())

-8.5
48.1


In [27]:
df['MaxTemp'] = df.groupby('Location')['MaxTemp'].transform(lambda x: x.fillna(x.median()))
df['MaxTemp'].isna().sum()

np.int64(0)

In [28]:
df['MinTemp'] = df.groupby('Location')['MaxTemp'].transform(lambda x: x.fillna(x.median()))
df['MinTemp'].isna().sum()

np.int64(0)

In [29]:
df["Rainfall"].isna().sum()

np.int64(0)

In [30]:
df['RainToday'].isna().sum()

np.int64(0)

In [31]:
df['RainToday'].value_counts()

RainToday
No     77428
Yes    22056
Name: count, dtype: int64

In [32]:
df.loc[(df['Rainfall'].isna()) & (df['RainToday'] == 'No'), 'Rainfall'] = 0

In [33]:
df.isna().sum()

Location             0
MinTemp              0
MaxTemp              0
Rainfall             0
Evaporation      42499
Sunshine         47285
WindGustDir          0
WindGustSpeed        0
WindDir9am           0
WindDir3pm           0
WindSpeed9am         0
WindSpeed3pm         0
Humidity9am          0
Humidity3pm          0
Pressure9am          0
Pressure3pm          0
Cloud9am             0
Cloud3pm             0
Temp9am              0
Temp3pm              0
RainToday            0
RainTomorrow         0
dtype: int64

In [34]:
df['Rainfall'] = df.groupby('Location')['Rainfall'].transform(lambda x: x.fillna(x.median()))

In [35]:
df['Evaporation'].value_counts()

Evaporation
4.000000     2296
8.000000     1764
2.200000     1453
2.000000     1406
2.400000     1403
             ... 
40.799999       1
43.200001       1
42.799999       1
33.000000       1
39.599998       1
Name: count, Length: 327, dtype: int64

In [36]:
(df["Evaporation"].isna().sum() / len(df)) * 100

np.float64(42.71943227051586)

In [37]:
df['Sunshine'].nunique()

145

In [38]:
df['Sunshine'].value_counts()

Sunshine
0.0     1626
10.7     771
11.0     762
10.8     754
10.5     719
        ... 
14.0      11
14.1       4
14.2       2
14.5       1
14.3       1
Name: count, Length: 145, dtype: int64

In [39]:
(df['Sunshine'].isna().sum() / len(df)) * 100

np.float64(47.530256121587385)

In [40]:
df['WindGustDir'].value_counts()

WindGustDir
W      11961
SE      6571
E       6556
SSE     6455
N       6418
SW      6276
S       6266
WSW     6180
SSW     6051
NW      5833
WNW     5752
ENE     5613
NE      5178
ESE     5130
NNE     4625
NNW     4619
Name: count, dtype: int64

In [41]:
(df['WindGustDir'].isna().sum() / len(df)) * 100

np.float64(0.0)

In [42]:
df.groupby('Location')['WindGustDir'].apply(lambda x: x.isna().all())

Location
Adelaide            False
Albany              False
Albury              False
AliceSprings        False
BadgerysCreek       False
Ballarat            False
Bendigo             False
Brisbane            False
Cairns              False
Canberra            False
Cobar               False
CoffsHarbour        False
Dartmoor            False
Darwin              False
GoldCoast           False
Hobart              False
Katherine           False
Launceston          False
Melbourne           False
MelbourneAirport    False
Mildura             False
Moree               False
MountGambier        False
MountGinini         False
Newcastle           False
Nhil                False
NorahHead           False
NorfolkIsland       False
Nuriootpa           False
PearceRAAF          False
Penrith             False
Perth               False
PerthAirport        False
Portland            False
Richmond            False
Sale                False
SalmonGums          False
Sydney              False
Syd

In [43]:
df['WindGustDir'] = df.groupby('Location')['WindGustDir'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x)
)

global_mode = df['WindGustDir'].mode()[0]
df['WindGustDir'] = df['WindGustDir'].fillna(global_mode)

In [44]:
df['WindGustDir'] = df['WindGustDir'].astype('category')

In [45]:
(df['WindGustSpeed'].isna().sum() / len(df)) * 100

np.float64(0.0)

In [46]:
df['WindGustSpeed'].value_counts()

WindGustSpeed
39.0     10833
35.0      6449
41.0      5958
31.0      5868
37.0      5865
         ...  
111.0        1
117.0        1
122.0        1
130.0        1
6.0          1
Name: count, Length: 67, dtype: int64

In [47]:
print(df['WindGustSpeed'].min())
print(df['WindGustSpeed'].max())

6.0
135.0


In [48]:
df['WindGustSpeed'] = df.groupby('Location')['WindGustSpeed'].transform(lambda x: x.fillna(df['WindGustSpeed'].median()))
df['WindGustSpeed'].isna().sum()

np.int64(0)

In [49]:
df['WindGustSpeed'] = df['WindGustSpeed'].astype('float32')

In [50]:
(df['WindDir9am'].isna().sum() / len(df)) * 100

np.float64(0.0)

In [51]:
df['WindDir9am'].value_counts()

WindDir9am
N      8861
NW     7650
SE     7005
E      6631
SSE    6458
SW     6421
NNW    6204
SSW    6136
S      6096
W      6012
NNE    5676
ENE    5518
ESE    5479
NE     5445
WNW    5097
WSW    4795
Name: count, dtype: int64

In [52]:
df['WindDir9am'] = df.groupby('Location')['WindDir9am'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x)
)

df['WindDir9am'].isna().sum()

np.int64(0)

In [53]:

(df['WindDir3pm'].isna().sum() / len(df)) * 100

np.float64(0.0)

In [54]:
df['WindDir3pm'] = df.groupby('Location')['WindDir3pm'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x)
)

df['WindDir3pm'].isna().sum()

np.int64(0)

In [55]:
df.isna().sum()

Location             0
MinTemp              0
MaxTemp              0
Rainfall             0
Evaporation      42499
Sunshine         47285
WindGustDir          0
WindGustSpeed        0
WindDir9am           0
WindDir3pm           0
WindSpeed9am         0
WindSpeed3pm         0
Humidity9am          0
Humidity3pm          0
Pressure9am          0
Pressure3pm          0
Cloud9am             0
Cloud3pm             0
Temp9am              0
Temp3pm              0
RainToday            0
RainTomorrow         0
dtype: int64

In [56]:
       
for col in ['WindSpeed9am', 'WindSpeed3pm']:
    df[col] = df.groupby('Location')[col].transform(lambda x: x.fillna(df[col].median()))
    print(df[col].isna().sum())

0
0


In [57]:
df.isna().sum()

Location             0
MinTemp              0
MaxTemp              0
Rainfall             0
Evaporation      42499
Sunshine         47285
WindGustDir          0
WindGustSpeed        0
WindDir9am           0
WindDir3pm           0
WindSpeed9am         0
WindSpeed3pm         0
Humidity9am          0
Humidity3pm          0
Pressure9am          0
Pressure3pm          0
Cloud9am             0
Cloud3pm             0
Temp9am              0
Temp3pm              0
RainToday            0
RainTomorrow         0
dtype: int64

In [58]:
df['Humidity3pm'].nunique()

101

In [59]:
       
for col in ['Humidity3pm', 'Humidity9am']:
    df[col] = df.groupby('Location')[col].transform(lambda x: x.fillna(df[col].median()))
    print(df[col].isna().sum())

0
0


In [60]:
df['Pressure3pm'].value_counts()

Pressure3pm
1015.299988    8728
1014.500000    1249
1015.900024     743
1016.099976     633
1015.799988     629
               ... 
990.500000        1
1039.599976       1
986.799988        1
1038.400024       1
989.500000        1
Name: count, Length: 537, dtype: int64

In [61]:
for col in ['Pressure9am', 'Pressure3pm']:
    df[col] = df.groupby('Location')[col].transform(lambda x: x.fillna(df[col].median()))
    print(df[col].isna().sum())

0
0


In [62]:
df.isna().sum()

Location             0
MinTemp              0
MaxTemp              0
Rainfall             0
Evaporation      42499
Sunshine         47285
WindGustDir          0
WindGustSpeed        0
WindDir9am           0
WindDir3pm           0
WindSpeed9am         0
WindSpeed3pm         0
Humidity9am          0
Humidity3pm          0
Pressure9am          0
Pressure3pm          0
Cloud9am             0
Cloud3pm             0
Temp9am              0
Temp3pm              0
RainToday            0
RainTomorrow         0
dtype: int64

In [63]:
df['Cloud3pm'].value_counts()

Cloud3pm
5    31460
7    18694
1    10294
6     9571
8     9417
2     5695
4     5498
3     5445
0     3410
Name: count, dtype: Int64

In [64]:
df['Cloud3pm'].isna().sum() / len(df)

np.float64(0.0)

In [65]:
df.loc[df["Cloud3pm"] == 9]

,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow


In [66]:
df.loc[df['Cloud3pm'] == 9.0, 'Cloud3pm'] = np.nan

In [67]:
df.isna().sum()

Location             0
MinTemp              0
MaxTemp              0
Rainfall             0
Evaporation      42499
Sunshine         47285
WindGustDir          0
WindGustSpeed        0
WindDir9am           0
WindDir3pm           0
WindSpeed9am         0
WindSpeed3pm         0
Humidity9am          0
Humidity3pm          0
Pressure9am          0
Pressure3pm          0
Cloud9am             0
Cloud3pm             0
Temp9am              0
Temp3pm              0
RainToday            0
RainTomorrow         0
dtype: int64

In [68]:
for col in ['Temp9am', 'Temp3pm']: 
    print((df[col].isna().sum() / len(df)) * 100)

0.0
0.0


In [69]:
df['Temp3pm'].value_counts()

Temp3pm
 22.000000    1134
 18.400000    1075
 18.500000     609
 19.200001     598
 19.000000     592
              ... 
-3.900000        1
-2.400000        1
-4.200000        1
-3.800000        1
 43.799999       1
Name: count, Length: 492, dtype: int64

In [70]:
df.info()

<class 'pandas.DataFrame'>
Index: 99484 entries, 0 to 99515
Data columns (total 22 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   Location       99484 non-null  category
 1   MinTemp        99484 non-null  float32 
 2   MaxTemp        99484 non-null  float32 
 3   Rainfall       99484 non-null  float32 
 4   Evaporation    56985 non-null  float32 
 5   Sunshine       52199 non-null  float32 
 6   WindGustDir    99484 non-null  category
 7   WindGustSpeed  99484 non-null  float32 
 8   WindDir9am     99484 non-null  category
 9   WindDir3pm     99484 non-null  category
 10  WindSpeed9am   99484 non-null  float32 
 11  WindSpeed3pm   99484 non-null  float32 
 12  Humidity9am    99484 non-null  float32 
 13  Humidity3pm    99484 non-null  float32 
 14  Pressure9am    99484 non-null  float32 
 15  Pressure3pm    99484 non-null  float32 
 16  Cloud9am       99484 non-null  Int8    
 17  Cloud3pm       99484 non-null  Int8    
 18  Te

In [71]:
for col in ['Temp9am', 'Temp3pm']: 
    df[col] = df[col].astype('float32')

In [72]:

for col in ['Temp9am', 'Temp3pm']: 
    df[col] = df.groupby('Location')[col].transform(lambda x: x.fillna(df[col].median()))
    print(df[col].isna().sum())

0
0


In [73]:
df.isna().sum()

Location             0
MinTemp              0
MaxTemp              0
Rainfall             0
Evaporation      42499
Sunshine         47285
WindGustDir          0
WindGustSpeed        0
WindDir9am           0
WindDir3pm           0
WindSpeed9am         0
WindSpeed3pm         0
Humidity9am          0
Humidity3pm          0
Pressure9am          0
Pressure3pm          0
Cloud9am             0
Cloud3pm             0
Temp9am              0
Temp3pm              0
RainToday            0
RainTomorrow         0
dtype: int64

In [74]:
df['RainToday'].value_counts()

RainToday
No     77428
Yes    22056
Name: count, dtype: int64

In [75]:
(df['RainToday'].isna().sum() / len(df)) * 100

np.float64(0.0)

In [76]:
df['RainToday'] = df.groupby('Location')['RainToday'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x)
)
(df['RainToday'].isna().sum() / len(df)) * 100

np.float64(0.0)

In [77]:
df.isna().sum()

Location             0
MinTemp              0
MaxTemp              0
Rainfall             0
Evaporation      42499
Sunshine         47285
WindGustDir          0
WindGustSpeed        0
WindDir9am           0
WindDir3pm           0
WindSpeed9am         0
WindSpeed3pm         0
Humidity9am          0
Humidity3pm          0
Pressure9am          0
Pressure3pm          0
Cloud9am             0
Cloud3pm             0
Temp9am              0
Temp3pm              0
RainToday            0
RainTomorrow         0
dtype: int64

In [78]:
import os

print(os.listdir(f"{BASE_PATH}/data/processed"))

['clean_weather_training_data.parquet']


In [79]:
from pathlib import Path

BASE_PATH = Path.cwd().parent

print(BASE_PATH)

processed_path = BASE_PATH / "data" / "processed"
processed_path.mkdir(parents=True, exist_ok=True)

print(processed_path)

d:\Aus-weather-analysis
d:\Aus-weather-analysis\data\processed


In [80]:
from pathlib import Path

processed_path = Path(BASE_PATH) / "data" / "processed"

processed_path.mkdir(parents=True, exist_ok=True)

output_file = processed_path / "clean_weather_training_data.parquet"

df.to_parquet(output_file, engine="pyarrow")

print(output_file)
print(output_file.exists())

d:\Aus-weather-analysis\data\processed\clean_weather_training_data.parquet
True


In [81]:
import os

print(os.listdir(f"{BASE_PATH}/data/processed"))

['clean_weather_training_data.parquet']


In [82]:
print(df.isnull().sum().sum())
print(df.shape)

89784
(99484, 22)


In [83]:
output_file = processed_path / "clean_weather_training_data.parquet"

df.to_parquet(output_file, engine="pyarrow")

print(output_file)
print(output_file.exists())

d:\Aus-weather-analysis\data\processed\clean_weather_training_data.parquet
True


In [84]:
import os

print(os.listdir(f"{BASE_PATH}/data/processed"))

['clean_weather_training_data.parquet']


In [85]:
print(df.shape)

print(df.columns.tolist())

print(df.isna().sum())

print(df.head())

(99484, 22)
['Location', 'MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustDir', 'WindGustSpeed', 'WindDir9am', 'WindDir3pm', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm', 'RainToday', 'RainTomorrow']
Location             0
MinTemp              0
MaxTemp              0
Rainfall             0
Evaporation      42499
Sunshine         47285
WindGustDir          0
WindGustSpeed        0
WindDir9am           0
WindDir3pm           0
WindSpeed9am         0
WindSpeed3pm         0
Humidity9am          0
Humidity3pm          0
Pressure9am          0
Pressure3pm          0
Cloud9am             0
Cloud3pm             0
Temp9am              0
Temp3pm              0
RainToday            0
RainTomorrow         0
dtype: int64
  Location    MinTemp    MaxTemp  Rainfall  Evaporation  Sunshine WindGustDir  \
0   Albury  22.900000  22.900000       0.6          NaN       NaN           W   
1   Al

In [86]:
print(df.columns.tolist())

df.to_parquet(
    f"{BASE_PATH}/data/processed/clean_weather_training_data.parquet",
    engine="pyarrow",
    index=False
)

['Location', 'MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustDir', 'WindGustSpeed', 'WindDir9am', 'WindDir3pm', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm', 'RainToday', 'RainTomorrow']


In [87]:
print(df.columns.tolist())

['Location', 'MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustDir', 'WindGustSpeed', 'WindDir9am', 'WindDir3pm', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm', 'RainToday', 'RainTomorrow']
